In [7]:
import nltk

def ngrams(sentence, n):
    words = sentence.split()
    ngrams = zip(*[words[i:] for i in range(n)])
    return list(ngrams)
sentence = '안녕하세요. 만나서 진심으로 반가워요'

unigram = ngrams(sentence,1)
bigram = ngrams(sentence,2)
trigram = ngrams(sentence,3)

print(unigram)
print(bigram)
print(trigram)

unigram = nltk.ngrams(sentence.split(), 1)
bigram = nltk.ngrams(sentence.split(), 2)
trigram = nltk.ngrams(sentence.split(), 3)

print(list(unigram))
print(list(bigram))
print(list(trigram))

[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요')]
[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요')]


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = ['That movie is famous movie',
          'I like that actor',
          "I don't like that actor"]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)
tfidf_matrix = tfidf_vectorizer.transform(corpus)

print(tfidf_matrix.toarray())
print(tfidf_vectorizer.vocabulary_)

[[0.         0.         0.39687454 0.39687454 0.         0.79374908
  0.2344005 ]
 [0.61980538 0.         0.         0.         0.61980538 0.
  0.48133417]
 [0.4804584  0.63174505 0.         0.         0.4804584  0.
  0.37311881]]
{'that': 6, 'movie': 5, 'is': 3, 'famous': 2, 'like': 4, 'actor': 0, 'don': 1}


In [9]:
import torch.nn as nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size,
                 embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                                      embedding_dim=embedding_dim)
        self.linear = nn.Linear(in_features=embedding_dim,
                                out_features=vocab_size)
    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [10]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"

import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\KDS10\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\KDS

In [11]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [12]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus=tokens,
                    n_vocab=5000,
                    special_tokens=['<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [13]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx-window_size)
            window_end = min(sentence_length,
                             idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx + 1:window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs
word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [14]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id['<unk>']
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [15]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:, 0]
context_indexes = index_pairs[:,1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset,
                        batch_size=32,
                        shuffle=True)

In [16]:
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
word2vec = VanillaSkipgram(vocab_size=len(token_to_id),
                           embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

cuda


In [17]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss.item()
    cost = cost / len(dataloader)
    print(epoch + 1, cost)

1 6.199500442452613
2 5.982251342579323
3 5.932418048570335
4 5.902017888885436
5 5.879464521640325
6 5.8619049384216435
7 5.846871398360618
8 5.833875714165426
9 5.822169800420706
10 5.811758301622655


In [18]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding
index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[-0.74208313 -0.24461202  0.40875244 -0.3122933   0.65103453  0.5106865
 -0.22170533 -0.5732475  -1.2644013   0.95621383 -0.7342277  -0.41154766
 -0.7903573  -0.3624901  -0.4530681  -0.4133717   1.8408904  -0.43313837
 -0.9259741  -1.12426     0.36578125 -0.6575514  -2.2527494  -0.98543787
 -1.203044   -0.6002799   0.9715791  -0.33536944 -0.34349713 -1.5063388
  0.57892764 -1.3092834  -0.2695192   0.8959102   0.0889376   0.8418631
  0.52054644 -0.9040324  -1.7415293   0.8981352   0.57277954 -1.3204339
  0.5891119   1.6538848   0.11204351  0.37608546  0.06447579 -0.28711626
  1.2122531   0.90366155 -0.5212376  -1.301522    0.6948762   0.6500822
 -0.25327864 -0.34951675 -0.7288804  -0.31019688  0.27105153  0.35197654
  1.2652165  -0.19418472 -0.54324514 -0.2573818  -0.68216    -0.5810774
 -0.14476341  1.1196388  -0.34383562 -1.4997534   0.14688253  0.590851
  0.37635392  0.81858325  0.95473063 -0.6591767   0.715139    1.3056204
 -1.1254091   0.47514802 -0.3571493   0.5823193  -0.01686

In [19]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1:n+1]
    return top_n
cosine_matrix = cosine_similarity(token_embedding,
                                  embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

for index in top_n:
    print(id_to_token[index], cosine_matrix[index])

오유 0.29042917
알겠는데 0.28116512
배우 0.27927494
~~~~ 0.27862725
조화 0.27506596
